# 🔋 STAGE-6: Meta-Learner Eğitimi
**Model:** `sklearn.linear_model.QuantileRegressor` (HiGHS LP, Karar 2)

**Girdi:** `x_meta_v2.joblib` — 1,051,950 × 9 OOF tahmin matrisi  
**Çıktı:** `meta_models_v2.joblib` — q=0.1, 0.5, 0.9 için 3 eğitilmiş model

---
### ✅ Çalıştırma Sırası
1. **Hücre 1** → Paket kurulumu
2. **Hücre 2** → Google Drive bağlantısı
3. **Hücre 3** → GitHub'dan kaynak kod çekme
4. **Hücre 4** → Veriyi Drive'dan kopyala
5. **Hücre 5** → Meta-learner eğitimi
6. **Hücre 6** → Sonuçları Drive'a kaydet
7. **Hücre 7** → Doğrulama & metrikler

In [ ]:
# ─── Hücre 1: Paket Kurulumu ───────────────────────────────────────────────
# Colab zaten sklearn içeriyor, sadece versiyonu kontrol et
import sklearn, numpy, pandas, joblib
print(f'sklearn : {sklearn.__version__}  (≥1.4 gerekli)')
print(f'numpy   : {numpy.__version__}')
print(f'pandas  : {pandas.__version__}')
print(f'joblib  : {joblib.__version__}')

# HiGHS LP solver kontrolü (sklearn>=1.4 ile geliyor)
from sklearn.linear_model import QuantileRegressor
import numpy as np
qr_test = QuantileRegressor(quantile=0.5, solver='highs')
qr_test.fit(np.array([[1],[2],[3]]), np.array([1,2,3]))
print('\n✅ HiGHS LP solver çalışıyor.')

In [ ]:
# ─── Hücre 2: Google Drive Bağlantısı ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
# Drive'daki klasör yolunu ayarla
# ⚠️  BURAYA KENDİ KLASÖR YOLUNU YAZ  ⚠️
# Örnek: /content/drive/MyDrive/tez-pv
DRIVE_DIR = '/content/drive/MyDrive/tez-pv'

os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive klasörü: {DRIVE_DIR}')
print('İçindeki dosyalar:')
for f in os.listdir(DRIVE_DIR):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')

In [ ]:
# ─── Hücre 3: GitHub'dan Kaynak Kod ───────────────────────────────────────
import subprocess

REPO_URL = 'https://github.com/arslanburak58/tez-pv-v2.git'
REPO_DIR = '/content/tez-pv-ant'

if not os.path.exists(REPO_DIR):
    print('Repo klonlanıyor...')
    subprocess.run(['git', 'clone', '--depth=1', REPO_URL, REPO_DIR], check=True)
else:
    print('Repo mevcut, güncelleniyor...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

# Python path'e ekle
import sys
sys.path.insert(0, REPO_DIR)

print(f'\n✅ Repo hazır: {REPO_DIR}')
print('models/meta_learner.py:', os.path.exists(f'{REPO_DIR}/models/meta_learner.py'))

In [ ]:
# ─── Hücre 4: Veriyi Drive'dan Kopyala ────────────────────────────────────
import shutil, pathlib

PROC_DIR = pathlib.Path(REPO_DIR) / 'data' / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Gerekli dosyalar
files_needed = {
    'x_meta_v2.joblib'    : '108 MB  — OOF meta-input matrisi',
    'base_models_v2.joblib': '3.7 MB  — 9 fully-trained base model (kontrol için)',
}

for fname, desc in files_needed.items():
    src = pathlib.Path(DRIVE_DIR) / fname
    dst = PROC_DIR / fname
    if dst.exists():
        print(f'✅ {fname} zaten mevcut ({dst.stat().st_size/1024**2:.1f} MB)')
    elif src.exists():
        print(f'⬇️  Kopyalanıyor: {fname}  ({desc})')
        shutil.copy2(str(src), str(dst))
        print(f'   ✅ Tamamlandı: {dst.stat().st_size/1024**2:.1f} MB')
    else:
        print(f'❌ HATA: {src} bulunamadı!')
        print(f'   → Lütfen {fname} dosyasını Drive/{DRIVE_DIR.split("/")[-1]}/ klasörüne yükleyin.')

print('\n--- Hazır Dosyalar ---')
for f in PROC_DIR.iterdir():
    if f.suffix in ['.joblib', '.parquet']:
        print(f'  {f.name:40s} {f.stat().st_size/1024**2:.1f} MB')

In [ ]:
# ─── Hücre 5: Meta-Learner Eğitimi ────────────────────────────────────────
import logging, time
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import QuantileRegressor

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger(__name__)

# 1. OOF matrisini yükle
x_meta_path = PROC_DIR / 'x_meta_v2.joblib'
logger.info(f'Yükleniyor: {x_meta_path}')
bundle = joblib.load(str(x_meta_path))

x_meta = bundle['x_meta']
y_meta = bundle['y_meta']

logger.info(f'x_meta shape : {x_meta.shape}')
logger.info(f'y_meta shape : {y_meta.shape}')
logger.info(f'NaN kontrol  : x={x_meta.isna().sum().sum()}, y={y_meta.isna().sum()}')

# 2. Eğitim
QUANTILES = [0.1, 0.5, 0.9]
models = {}
coef_table = []

for q in QUANTILES:
    t0 = time.time()
    logger.info(f'Eğitiliyor: QuantileRegressor(q={q}, solver=highs, alpha=0.0)')
    qr = QuantileRegressor(quantile=q, alpha=0.0, solver='highs')
    qr.fit(x_meta, y_meta)
    elapsed = time.time() - t0
    models[q] = qr
    coef_row = {'q': q, 'intercept': round(qr.intercept_, 5)}
    coef_row.update({col: round(c, 5) for col, c in zip(x_meta.columns, qr.coef_)})
    coef_table.append(coef_row)
    logger.info(f'  → q={q} tamamlandı ({elapsed:.1f}s)')

print('\n=== Meta-Learner Katsayı Tablosu ===')
coef_df = pd.DataFrame(coef_table).set_index('q')
print(coef_df.to_string())

In [ ]:
# ─── Hücre 6: OOF Metrikler & Monotonicity Kontrolü ──────────────────────

def compute_pinball(y_true, y_pred, q):
    r = y_true - y_pred
    return float(np.mean(np.where(r >= 0, q * r, (q - 1.0) * r)))

y_arr = y_meta.values

# OOF tahminleri
preds_raw = pd.DataFrame(
    {f'q_{q}': models[q].predict(x_meta) for q in QUANTILES},
    index=x_meta.index
)

# Post-sort (Chernozhukov 2010)
vals_sorted = np.sort(preds_raw.values, axis=1)
preds_sorted = pd.DataFrame(vals_sorted, columns=preds_raw.columns, index=preds_raw.index)

print('=== OOF Pinball Loss (post-sort sonrası) ===')
for q in QUANTILES:
    col = f'q_{q}'
    pb_raw    = compute_pinball(y_arr, preds_raw[col].values, q)
    pb_sorted = compute_pinball(y_arr, preds_sorted[col].values, q)
    print(f'  q={q}  raw={pb_raw:.6f}  post-sort={pb_sorted:.6f}')

# PICP
lo = preds_sorted['q_0.1'].values
hi = preds_sorted['q_0.9'].values
picp = float(((y_arr >= lo) & (y_arr <= hi)).mean())
print(f'\nPICP [0.1–0.9] : {picp:.4f}  (hedef: ~0.80, nominal=0.80)')
print(f'PICP yorumu    : {"✅ Kalibre" if 0.75 <= picp <= 0.85 else "⚠️  Kontrol et"}')

# Crossing kontrolü
cross_after = ((preds_sorted['q_0.1'] > preds_sorted['q_0.5']) | (preds_sorted['q_0.5'] > preds_sorted['q_0.9'])).sum()
print(f'\nPost-sort crossing: {cross_after}  (0 olmalı ✅)')

# Özet
metrics = {
    'pinball_q10': compute_pinball(y_arr, preds_sorted['q_0.1'].values, 0.1),
    'pinball_q50': compute_pinball(y_arr, preds_sorted['q_0.5'].values, 0.5),
    'pinball_q90': compute_pinball(y_arr, preds_sorted['q_0.9'].values, 0.9),
    'picp_80'    : picp,
}

In [ ]:
# ─── Hücre 7: Sonuçları Drive'a Kaydet ────────────────────────────────────
import json

# 1. meta_models_v2.joblib — Colab'a kaydet
out_colab = PROC_DIR / 'meta_models_v2.joblib'
payload = {
    'models'      : models,
    'quantiles'   : QUANTILES,
    'columns'     : list(x_meta.columns),
    'oof_metrics' : metrics,
    'coef_df'     : coef_df,
}
joblib.dump(payload, str(out_colab), compress=3)
print(f'✅ Colab: {out_colab}  ({out_colab.stat().st_size/1024:.1f} KB)')

# 2. Drive'a kopyala
out_drive = pathlib.Path(DRIVE_DIR) / 'meta_models_v2.joblib'
shutil.copy2(str(out_colab), str(out_drive))
print(f'✅ Drive: {out_drive}  ({out_drive.stat().st_size/1024:.1f} KB)')

# 3. Metrik özeti (JSON) — Drive'a kaydet
metrics_path = pathlib.Path(DRIVE_DIR) / 'stage6_metrics.json'
with open(str(metrics_path), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'✅ Metrikler: {metrics_path}')

print('\n=== STAGE-6 TAMAMLANDI ===')
print(f'Pinball q10 : {metrics["pinball_q10"]:.6f}')
print(f'Pinball q50 : {metrics["pinball_q50"]:.6f}')
print(f'Pinball q90 : {metrics["pinball_q90"]:.6f}')
print(f'PICP 80%    : {metrics["picp_80"]:.4f}')
print('\n⬇️  meta_models_v2.joblib Drive klasörünüzde hazır.')
print('    Kendi bilgisayarınıza indirip data/processed/ klasörüne koyun.')

## 📋 Notebooktan Sonra Yapılacaklar

1. Drive klasöründeki `meta_models_v2.joblib` dosyasını bilgisayarınıza indirin
2. `tez-pv-ant/data/processed/meta_models_v2.joblib` olarak kaydedin
3. Antigravity'e dönerek STAGE-7'ye geçin

---
**Referans:**  
Koenker, R. & Bassett, G. (1978). Regression quantiles. *Econometrica*, 46(1), 33–50.  
Chernozhukov, V. et al. (2010). Quantile and probability curves without crossing. *Econometrica*, 78(3), 1093–1125.